# Czech UGS Storage Target — Season-Long Fill Requirement

**Question this notebook answers**
What working-gas volume must Czech UGS hold on **1 October** to survive a
1-in-20 cold-winter demand event, at each import-risk scenario (S1–S5)?

**Method**
A daily draw-down simulation runs 1 October → 31 March (182 days).
Each day uses the full monthly 1-in-20 peak demand — a conservative framing
(all days at peak; no 7+23 split used in Branch 1).  Bisection finds the
minimum 1-October fill at which the simulation remains feasible.

**Key finding**
The binding constraint is withdrawal *rate* (S1) or volume (S2–S5) — see results.
A TWh-only obligation without a corresponding rate check is incomplete.

*All analytical logic lives in `bsd`; this notebook calls the library and renders results.*

---
## 0. Setup

In [ ]:
%cd ..
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import bsd
import bsd.jvs as jvs

jvs.apply_style(theme="light", context="notebook")
%matplotlib inline

SCENARIOS = bsd.DEFAULT_SCENARIOS_DICT

# ── Withdrawal curve (Branch 2: 75/25 blend, 10% haircut) ──
WC_ASSUMPTION       = "blend"
WC_BLEND_WEIGHT_ENG = 0.75   # Branch 2 default — see docs/wc_blend_decision.md
IMPORT_ASSUMPTION   = "p99_cold"

wc_curves = bsd.fit_withdrawal_curves()
wc_func = wc_curves.blend(eng_weight=WC_BLEND_WEIGHT_ENG)

print(f"WC assumption: {WC_ASSUMPTION}  (eng_weight={WC_BLEND_WEIGHT_ENG:.0%})")
print(f"WC at 20%: {wc_func(20):.1f}  50%: {wc_func(50):.1f}  80%: {wc_func(80):.1f}  GWh/d")

---
## 1. Peak demand

In [ ]:
prof = bsd.load_demand_profile()
peak_demand = prof.peak  # Season simulation uses peak demand every day
peak_demand.rename(bsd.MONTH_NAMES)

---
## 2. Monthly import percentiles per scenario

In [ ]:
monthly_imports = bsd.compute_monthly_imports(SCENARIOS)
p99_cold, p99_uncond = bsd.compute_p99_imports()

IMPORT_CEILING = {"p99_cold": p99_cold,
                  "p99_unconditional": p99_uncond}[IMPORT_ASSUMPTION]

imp_df = bsd.import_table(monthly_imports, p99_cold, p99_uncond)
print(f"Active ceiling assumption: {IMPORT_ASSUMPTION}")
imp_df

---
## 3. Withdrawal-curve reference

75/25 blend (engineering/empirical) — higher engineering weight justified because the
season-long simulation regularly reaches fill <20% where empirical data are sparse.
See `docs/wc_blend_decision.md`.

In [ ]:
print(f"WC_CURRENT: {wc_curves.wc_current:.1f} GWh/d")
for fp in [20, 50, 80]:
    print(f"  {fp}%: blend={wc_func(fp):.1f}  "
          f"empirical={wc_curves.empirical(fp):.1f}  "
          f"engineering={wc_curves.engineering(fp):.1f}  GWh/d")

---
## 4. Main results — minimum 1-October fill

`bsd.run_all_scenarios` calls `bsd.min_start_fill` (bisection) and `bsd.simulate`
for each scenario.

In [ ]:
main_results = bsd.run_all_scenarios(
    monthly_imports, wc_func, peak_demand, SCENARIOS,
)

for key, sc in SCENARIOS.items():
    info = main_results[key]
    f = info["start_fill_pct"]
    if f is None:
        print(f"{key}: INFEASIBLE at 100% start fill")
    else:
        sim = info["sim"]
        print(f"{key}: start fill = {f:5.2f}%  ({f * bsd.CAPACITY_TWH / 100:5.2f} TWh)  "
              f"min fill = {sim['min_fill_pct']:5.2f}%  binding = {info['binding']}")

In [ ]:
# Summary table
rows = []
for key, sc in SCENARIOS.items():
    info = main_results[key]
    f = info["start_fill_pct"]
    if f is None:
        rows.append({"Scenario": sc.label, "Start fill (%)": "infeasible",
                     "Start fill (TWh)": "\u2014", "Min fill reached (%)": "\u2014",
                     "Binding constraint": "infeasible at 100%"})
    else:
        rows.append({
            "Scenario":              sc.label,
            "Start fill (%)":        round(f, 2),
            "Start fill (TWh)":      round(f * bsd.CAPACITY_TWH / 100, 2),
            "Min fill reached (%)":  round(info["sim"]["min_fill_pct"], 2),
            "Binding constraint":    info["binding"],
        })
summary = pd.DataFrame(rows).set_index("Scenario")
summary

---
## 5. Sensitivity analysis

In [ ]:
sens_df = bsd.sensitivity_table(
    SCENARIOS, monthly_imports, wc_func, peak_demand,
    wc_abs_func=wc_curves.empirical_abs,
    peak_shifts=(0.0, +50.0, -50.0),
)
sens_df

---
## 6. Ceiling-import benchmark (P99, cold days)

In [ ]:
# All scenarios under ceiling imports
ceiling_imports = {k: IMPORT_CEILING for k in SCENARIOS}
ceiling_target = {}
for key, sc in SCENARIOS.items():
    f = bsd.min_start_fill(key, ceiling_imports, wc_func, peak_demand)
    ceiling_target[sc.label] = {
        "Start fill (TWh)": round(f * bsd.CAPACITY_TWH / 100, 2) if f is not None else None,
        "Start fill (%)":   round(f, 2) if f is not None else None,
    }

ceiling_df = pd.DataFrame(ceiling_target).T
print(f"Season target under {IMPORT_ASSUMPTION} ceiling imports (GWh/d):")
print(IMPORT_CEILING.rename(bsd.MONTH_NAMES).round(1).to_string())
print(f"\nvs. base S2 target: {main_results['S2']['start_fill_TWh']:.2f} TWh")
ceiling_df

---
## 7. Charts

In [ ]:
month_starts = np.cumsum([0] + [bsd.DAYS_IN_MONTH[m] for m in bsd.MONTH_ORDER[:-1]])

fig, ax = plt.subplots(figsize=(10, 5))
for key, sc in SCENARIOS.items():
    info = main_results[key]
    if info["sim"] is None: continue
    traj = info["sim"]["fill_trajectory"]
    ax.plot(range(len(traj)), traj, color=sc.color, lw=2,
            label=f"{sc.label}  start={info['start_fill_pct']:.1f}%")
ax.set_xticks(month_starts)
ax.set_xticklabels([bsd.MONTH_NAMES[m] for m in bsd.MONTH_ORDER])
ax.set_ylabel("Fill level (%)")
ax.set_title("Storage fill-level trajectory from minimum required start (Oct 1 \u2192 Mar 31)")
ax.axhline(0, color="grey", lw=0.8, ls=":")
ax.axhline(20, color="grey", lw=0.6, ls="--", alpha=0.6,
           label="Low-confidence WC regime (<20%)")
ax.legend(fontsize=8)
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())
fig.tight_layout()
fig.savefig("figs/storage_target_fill_trajectories.png", bbox_inches="tight")
fig;

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for key, sc in SCENARIOS.items():
    info = main_results[key]
    if info["sim"] is None: continue
    h = info["sim"]["headroom_trajectory"]
    ax.plot(range(len(h)), h, color=sc.color, lw=1.6, label=sc.label)
ax.axhline(0, color="red", lw=1, ls="--", label="Withdrawal-rate constraint binds")
ax.set_xticks(month_starts); ax.set_xticklabels([bsd.MONTH_NAMES[m] for m in bsd.MONTH_ORDER])
ax.set_ylabel("Headroom (max_wc \u2212 daily_gap), GWh/d")
ax.set_title("Withdrawal-rate headroom over the season")
ax.legend(fontsize=8)
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())
fig.tight_layout()
fig.savefig("figs/storage_target_headroom.png", bbox_inches="tight")
fig;

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
keys = list(SCENARIOS); xs = np.arange(len(keys))
twh = [main_results[k]["start_fill_TWh"] or 0 for k in keys]
colors = [SCENARIOS[k].color for k in keys]
ax.bar(xs, twh, color=colors, alpha=0.85)
ax.axhline(bsd.CAPACITY_TWH, color="black", lw=1.2, ls="--",
           label=f"Czech UGS capacity ({bsd.CAPACITY_TWH} TWh)")
for x, v in zip(xs, twh):
    ax.text(x, v + 0.4, f"{v:.2f}", ha="center", fontsize=9)
ax.set_xticks(xs); ax.set_xticklabels([SCENARIOS[k].label for k in keys], rotation=20, ha="right")
ax.set_ylabel("Minimum required start-of-winter fill (TWh)")
ax.set_title("Minimum 1-October fill to survive 1-in-20 winter, by scenario")
ax.legend(fontsize=8)
ax.set_ylim(0, bsd.CAPACITY_TWH * 1.05)
fig.tight_layout()
fig.savefig("figs/storage_target_bar.png", bbox_inches="tight")
fig;

---
## 8. Caveats

1. **Deterministic monthly imports.** Conservative upper-bound framing.
2. **Withdrawal curve assumption.** 75/25 blend motivated by sparse empirical data at <20% fill.
3. **Injection feasibility not checked.** Targets up to 25.2 TWh (S1) should be verified against
   Apr–Sep injection-rate constraints and EU filling obligations.
4. **Demand/import correlation.** Copula-based joint model recommended as next step.